<a href="https://colab.research.google.com/github/Soha-Waseem/FlyRank_ML_Week1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Soha-Waseem/FlyRank_ML_Week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
import os
import getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("Paste HF token: ")

In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [ ]:
df = con.sql(f"""
SELECT *
FROM {TABLES['fact_daily_sample']}
LIMIT 100000
""").df()

print(df.columns.tolist())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [ ]:
df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [ ]:
print(df.columns.tolist())


['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [ ]:
# Create CTR column

import numpy as np

df = df[
    (df["gsc_data_available"] == True) &
    (df["gsc_impressions"] > 0) &
    (df["gsc_avg_position"].notna())
].copy()

In [ ]:
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"]

**Signal 1:** CTR should generally decrease as average search position becomes worse. I grouped pages into position buckets and compared the average CTR for each bucket.

In [ ]:
# Verify Signal 1 (CTR vs Position)

df["position_bucket"] = pd.cut(
    df["gsc_avg_position"],
    bins=[0, 3, 10, 20, 50, 100],
    labels=[
        "Top 3",
        "4-10",
        "11-20",
        "21-50",
        "50+"
    ]
)
signal1 = (
    df.groupby("position_bucket")
      .agg(
          avg_ctr=("ctr", "mean"),
          n=("ctr", "count")
      )
)

signal1

/tmp/ipykernel_3037/1750701616.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("position_bucket")


,avg_ctr,n
position_bucket,,
Top 3,0.011164,2266
4-10,0.006138,12337
11-20,0.004354,4532
21-50,0.002022,4388
50+,0.000470,2512


**Verdict**: CONFIRMED

**Observed**: Pages with better average positions generally have higher CTR. This supports using CTR relative to position as a signal for optimization.

**Signal 2:** I checked whether higher-impression pages show meaningful CTR opportunities.

In [ ]:
# Verify Signal 2 (Search Volume)
df["impression_bucket"] = pd.qcut(
    df["gsc_impressions"],
    q=4,
    duplicates="drop"
)

signal2 = (
    df.groupby("impression_bucket")
      .agg(
          avg_ctr=("ctr", "mean"),
          n=("ctr", "count")
      )
)

signal2

/tmp/ipykernel_3037/3745440679.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("impression_bucket")


,avg_ctr,n
impression_bucket,,
"(0.999, 3.0]",0.006718,7840
"(3.0, 9.0]",0.004419,5991
"(9.0, 30.0]",0.004953,6202
"(30.0, 27135.0]",0.004828,6634


**Verdict**: MIXED

**Observed**: Higher-impression pages provide more optimization opportunities, but the relationship with CTR is not perfectly consistent across buckets.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule:** Prioritize pages with high search visibility but relatively low CTR. These pages already appear in search results, so improving titles and meta descriptions may increase clicks without requiring higher rankings.

**Reason code:** LOW_CTR

**Action:** FIX_TITLE_META

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["score"] = (
    np.log1p(df["gsc_impressions"]) * 10
    + (11 - df["gsc_avg_position"]).clip(lower=0) * 5
    - df["ctr"] * 100
)

df["reason_code"] = "LOW_CTR"
df["action"] = "FIX_TITLE_META"

queue = df.sort_values("score", ascending=False)

import os

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved:", os.path.exists("work/outputs/baseline_action_score.csv"))

Saved: True


The baseline score combines search visibility, ranking position, and click-through rate. Higher scores indicate pages that are visible in search but may benefit from improving titles and meta descriptions.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20)

top20[
    [
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
    ]
]

,content_hash_id,score,reason_code,action,gsc_impressions,gsc_clicks,ctr,gsc_avg_position
73504,content_eadb33b5df496f4a,145.955707,LOW_CTR,FIX_TITLE_META,27135,175,0.006449,2.097107
73493,content_545bb6cc7081ded3,143.387909,LOW_CTR,FIX_TITLE_META,18168,27,0.001486,1.907640
73491,content_9ef3d7516483e665,142.035288,LOW_CTR,FIX_TITLE_META,16473,22,0.001336,1.985309
73866,content_f416d3769e99bd5a,129.352935,LOW_CTR,FIX_TITLE_META,4737,0,0.000000,2.056154
73490,content_61215c724c8220ae,125.344698,LOW_CTR,FIX_TITLE_META,2975,15,0.005042,1.826891
75472,content_51fcf284e7e005c0,124.958860,LOW_CTR,FIX_TITLE_META,3364,5,0.001486,2.220868
73271,content_cc26620b2cbb837f,124.733536,LOW_CTR,FIX_TITLE_META,14001,22,0.001571,5.115777
73505,content_0e03de7680314cd5,123.752891,LOW_CTR,FIX_TITLE_META,3195,9,0.002817,2.332394
73149,content_21309e9a83c83653,121.805518,LOW_CTR,FIX_TITLE_META,8415,13,0.001545,4.683779
73429,content_8d7d99f109e19aa2,120.861205,LOW_CTR,FIX_TITLE_META,2555,6,0.002348,2.473190


### Top-20 Review

The baseline rule ranked pages based on high search visibility (impressions), relatively good average search position, and low CTR. These pages are considered candidates for title and meta description optimization because they already receive impressions but may attract more clicks.

| Rank | Action | Reason Code | Confidence Note | What would make it wrong |
|------|--------|-------------|-----------------|--------------------------|
| 1 | FIX_TITLE_META | LOW_CTR | Very high confidence. The page has 27,135 impressions, an average position of 2.10, but a CTR of only 0.64%, indicating a strong optimization opportunity. | The title or meta description may have been updated recently and the improvement is not yet reflected in the data. |
| 2 | FIX_TITLE_META | LOW_CTR | High confidence. The page ranks around position 1.91 with 18,168 impressions but has only 0.15% CTR. | SERP features such as featured snippets or ads may reduce CTR regardless of title quality. |
| 3 | FIX_TITLE_META | LOW_CTR | High confidence. Strong visibility (16,473 impressions) with only 0.13% CTR suggests room for improvement. | User search intent may already be satisfied without clicking the result. |
| 4 | FIX_TITLE_META | LOW_CTR | Medium confidence. The page receives 4,737 impressions but no recorded clicks despite ranking around position 2.06. | Data may represent a short reporting period or incomplete Search Console data. |
| 5 | FIX_TITLE_META | LOW_CTR | High confidence. Position 1.83 and nearly 3,000 impressions indicate visibility, while CTR remains relatively low. | Recent SEO changes may not yet appear in the available data. |
| 6 | FIX_TITLE_META | LOW_CTR | Medium confidence. Good ranking and more than 3,300 impressions indicate optimization potential. | Traffic could be seasonal or influenced by temporary search trends. |
| 7 | FIX_TITLE_META | LOW_CTR | High confidence. More than 14,000 impressions with modest CTR indicate a clear opportunity. | Competitors may have more attractive search snippets that limit achievable CTR. |
| 8 | FIX_TITLE_META | LOW_CTR | Medium confidence. The page has over 3,000 impressions and relatively low CTR despite a good position. | The query may naturally have a lower CTR because of SERP layout. |
| 9 | FIX_TITLE_META | LOW_CTR | High confidence. More than 8,000 impressions and low CTR indicate optimization potential. | Google may rewrite the page title in search results. |
| 10 | FIX_TITLE_META | LOW_CTR | Medium confidence. The page ranks well but CTR remains below expectations. | Search demand may fluctuate over time. |
| 11 | FIX_TITLE_META | LOW_CTR | Medium confidence. Good search visibility suggests improving metadata could increase clicks. | User behaviour may already be changing in newer data. |
| 12 | FIX_TITLE_META | LOW_CTR | Medium confidence. The page has a strong ranking with relatively low engagement. | Missing contextual information such as branded searches may affect interpretation. |
| 13 | FIX_TITLE_META | LOW_CTR | Medium confidence. Visibility is sufficient to justify optimization efforts. | CTR may already be improving after recent content updates. |
| 14 | FIX_TITLE_META | LOW_CTR | Medium confidence. Ranking near the top of search results indicates optimization potential. | Competitor SERP features may continue to suppress clicks. |
| 15 | FIX_TITLE_META | LOW_CTR | Medium confidence. Current metrics suggest room for CTR improvement. | Search Console reporting delays may affect the latest values. |
| 16 | FIX_TITLE_META | LOW_CTR | Medium confidence. Good ranking with relatively low CTR supports the recommendation. | Low traffic during the reporting period could reduce reliability. |
| 17 | FIX_TITLE_META | LOW_CTR | Medium confidence. Although impressions are lower, the page still appears to underperform in CTR. | Additional historical data may change the ranking. |
| 18 | FIX_TITLE_META | LOW_CTR | Medium confidence. The page shows measurable optimization opportunity. | Query intent may naturally produce fewer clicks. |
| 19 | FIX_TITLE_META | LOW_CTR | Medium confidence. The baseline score identifies the page as a reasonable optimization candidate. | The observed pattern may not remain stable over future reporting periods. |
| 20 | FIX_TITLE_META | LOW_CTR | Medium confidence. The page has sufficient visibility to justify investigation. | Additional business context or content quality considerations may change the recommendation. |

Overall, the ranked queue is intended as a decision-support baseline rather than a final recommendation. The rule uses only current Google Search Console metrics and does not rely on future information or labels.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

  - Recommendations with very low impressions or missing GSC values are less reliable.
  - The score uses only current GSC metrics (gsc_impressions, gsc_clicks, gsc_avg_position) and does not use future information, labels, or product flags, so it avoids data leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.